# Social Network Analysis — Merdeka Curriculum Discourse on X

Portfolio version of the **Social Network Analysis (SNA)** branch used in the published study **“Sentiment and Social Network Analysis of X Social Media on the Implementation of the Merdeka Belajar Curriculum in Indonesia”** (Journal of Advanced Technology and Multidiscipline, 2025).

The network analysis examines user interactions and identifies strategically positioned actors using:

- Degree centrality
- Betweenness centrality
- Closeness centrality
- Mention-based sentiment distribution

The sentiment and SNA pipelines were implemented separately in the published study and interpreted side by side.

> **Portfolio note:** Raw tweets, local paths, exported working files, and exploratory author-based sentiment summaries are excluded. The public notebook retains the mention-based analysis consistent with the published results.

---


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
# Import necessary libraries
import pandas as pd
import networkx as nx
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
# Load the dataset
file_path = 'path/to/social_network_dataset.xlsx'
df = pd.read_excel(file_path)
df.head()

In [ ]:
# Preprocessing functions
def preprocess_text(text):
    if not isinstance(text, str):  # Handle non-string inputs
        text = ""
    # Remove URLs
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = url_pattern.sub(r'', text)
    # Remove mentions
    text = re.sub(r'@[\w]*', ' ', text)
    # Remove punctuations
    punctuations = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    for x in text.lower():
        if x in punctuations:
            text = text.replace(x, " ")
    # Normalize text
    text = text.strip().lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('indonesian'))
    tokens = [word for word in tokens if word not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

# Apply preprocessing to the 'full_text' column
df['processed_text'] = df['full_text'].apply(preprocess_text)

# Load slang words dictionary
slang_url = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
slangs = pd.read_csv(slang_url)

# Function to replace slang words with their formal counterparts
def replace_slang(text):
    words = text.split()
    updated_words = [
        slangs.loc[slangs['slang'] == word, 'formal'].values[0]
        if word in slangs['slang'].values
        else word
        for word in words
    ]
    return " ".join(updated_words)

# Apply slang replacement
df['clean'] = df['processed_text'].apply(replace_slang)

In [ ]:
# Hapus username yang tidak diinginkan**
usernames_to_remove = ["mantanGL", "littlenazt"]
df = df[~df["username"].isin(usernames_to_remove)]

# Hapus baris dengan username kosong atau NaN
df = df.dropna(subset=["username"])
df = df[df["username"] != ""]

# Simpan data yang sudah dibersihkan**

In [ ]:
# Create 'mentions' column by extracting usernames mentioned in the tweet text
df['mentions'] = df['full_text'].apply(lambda x: re.findall(r'@(\w+)', str(x)))

# Create a directed graph for SNA
G = nx.DiGraph()

# Populate the graph with edges from mentions
for index, row in df.iterrows():
    user = row['username']  # Assumes a column 'username' exists
    mentions = row['mentions']
    for mention in mentions:
        G.add_edge(user, mention)

# Calculate centrality measures
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)

# Extract top 5 users based on each centrality measure
top_users_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
top_users_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
top_users_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]

# Format results for readability
degree_top5_df = pd.DataFrame(top_users_degree, columns=["Username", "Degree Centrality"])
betweenness_top5_df = pd.DataFrame(top_users_betweenness, columns=["Username", "Betweenness Centrality"])
closeness_top5_df = pd.DataFrame(top_users_closeness, columns=["Username", "Closeness Centrality"])

## Degree Centrality

In [ ]:
print("Top 5 Users by Degree Centrality:")
print(degree_top5_df)

In [ ]:
tweet_counts = {}
for username in degree_top5_df['Username']:
    tweet_counts[username] = len(df[df['username'] == username])

for username, count in tweet_counts.items():
    print(f"Jumlah tweet dari {username}: {count}")

In [ ]:
# Calculate mention counts for each user in degree_top5_df
mention_counts = {}
for username in degree_top5_df['Username']:
    mention_counts[username] = df['mentions'].apply(lambda mentions: username in mentions).sum()

for username, count in mention_counts.items():
    print(f"Jumlah mention dari {username}: {count}")


### Mention Analysis — nadiemmakarim

In [ ]:
def analyze_mentions(df, target_user):
    mentions_df = df[df['mentions'].apply(lambda mentions: target_user in mentions)]

    mentioning_users = mentions_df['username'].unique()

    user_sentiment = {}

    for user in mentioning_users:
        user_tweets = mentions_df[mentions_df['username'] == user]
        positive_count = 0
        negative_count = 0
        neutral_count = 0

        # You'll need to implement sentiment analysis here using a suitable library
        # This is a placeholder, replace with your sentiment analysis logic.
        for tweet in user_tweets['clean']:
          #Example, replace with your sentiment analysis model
            if "good" in tweet or "great" in tweet:
                positive_count += 1
            elif "bad" in tweet or "terrible" in tweet:
                negative_count += 1
            else:
                neutral_count += 1

        user_sentiment[user] = {
            'positive': positive_count,
            'negative': negative_count,
            'neutral': neutral_count
        }
    return user_sentiment

target_user = "nadiemmakarim"
sentiment_analysis_result = analyze_mentions(df, target_user)


top_mentioners = sorted(sentiment_analysis_result.items(), key=lambda x: sum(x[1].values()), reverse=True)[:3]


print(f"Top 3 users mentioning {target_user} and their sentiment analysis:")
for user, sentiment in top_mentioners:
    print(f"User: {user}")
    print(f"  Positive: {sentiment['positive']}")
    print(f"  Negative: {sentiment['negative']}")
    print(f"  Neutral: {sentiment['neutral']}")


### Mention Analysis — collegemenfess

In [ ]:
def analyze_mentions(df, target_user):
    mentions_df = df[df['mentions'].apply(lambda mentions: target_user in mentions)]

    mentioning_users = mentions_df['username'].unique()

    user_sentiment = {}

    for user in mentioning_users:
        user_tweets = mentions_df[mentions_df['username'] == user]
        positive_count = 0
        negative_count = 0
        neutral_count = 0

        # You'll need to implement sentiment analysis here using a suitable library
        # This is a placeholder, replace with your sentiment analysis logic.
        for tweet in user_tweets['clean']:
          #Example, replace with your sentiment analysis model
            if "good" in tweet or "great" in tweet:
                positive_count += 1
            elif "bad" in tweet or "terrible" in tweet:
                negative_count += 1
            else:
                neutral_count += 1

        user_sentiment[user] = {
            'positive': positive_count,
            'negative': negative_count,
            'neutral': neutral_count
        }
    return user_sentiment

target_user = "collegemenfess"
sentiment_analysis_result = analyze_mentions(df, target_user)


top_mentioners = sorted(sentiment_analysis_result.items(), key=lambda x: sum(x[1].values()), reverse=True)[:3]


print(f"Top 3 users mentioning {target_user} and their sentiment analysis:")
for user, sentiment in top_mentioners:
    print(f"User: {user}")
    print(f"  Positive: {sentiment['positive']}")
    print(f"  Negative: {sentiment['negative']}")
    print(f"  Neutral: {sentiment['neutral']}")


### Mention Analysis — Kemdikbud_RI

In [ ]:
def analyze_mentions(df, target_user):
    mentions_df = df[df['mentions'].apply(lambda mentions: target_user in mentions)]

    mentioning_users = mentions_df['username'].unique()

    user_sentiment = {}

    for user in mentioning_users:
        user_tweets = mentions_df[mentions_df['username'] == user]
        positive_count = 0
        negative_count = 0
        neutral_count = 0

        # You'll need to implement sentiment analysis here using a suitable library
        # This is a placeholder, replace with your sentiment analysis logic.
        for tweet in user_tweets['clean']:
          #Example, replace with your sentiment analysis model
            if "good" in tweet or "great" in tweet:
                positive_count += 1
            elif "bad" in tweet or "terrible" in tweet:
                negative_count += 1
            else:
                neutral_count += 1

        user_sentiment[user] = {
            'positive': positive_count,
            'negative': negative_count,
            'neutral': neutral_count
        }
    return user_sentiment

target_user = "Kemdikbud_RI"
sentiment_analysis_result = analyze_mentions(df, target_user)


top_mentioners = sorted(sentiment_analysis_result.items(), key=lambda x: sum(x[1].values()), reverse=True)[:3]


print(f"Top 3 users mentioning {target_user} and their sentiment analysis:")
for user, sentiment in top_mentioners:
    print(f"User: {user}")
    print(f"  Positive: {sentiment['positive']}")
    print(f"  Negative: {sentiment['negative']}")
    print(f"  Neutral: {sentiment['neutral']}")


In [ ]:
import pandas as pd
user_stats = {}

# Iterate over the top 5 users
for username in degree_top5_df['Username']:
    user_df = df[df['username'] == username]
    retweet_count = user_df['retweet_count'].sum()
    favorite_count = user_df['favorite_count'].sum()
    reply_count = user_df['reply_count'].sum()
    user_stats[username] = {
        'Retweets': retweet_count,
        'Favorites': favorite_count,
        'Replies': reply_count
    }

# Create a DataFrame from the user_stats dictionary
stats_df = pd.DataFrame.from_dict(user_stats, orient='index')

# Display the DataFrame
print(stats_df.astype(int))

## Betweenness Centrality



In [ ]:
print("\nTop 5 Users by Betweenness Centrality:")
print(betweenness_top5_df)

In [ ]:
tweet_counts = {}
for username in betweenness_top5_df['Username']:
    tweet_counts[username] = len(df[df['username'] == username])

for username, count in tweet_counts.items():
    print(f"Jumlah tweet dari {username}: {count}")

In [ ]:
# Calculate mention counts for each user in betweenness_top5_df
mention_counts = {}
for username in betweenness_top5_df['Username']:
    mention_counts[username] = df['mentions'].apply(lambda mentions: username in mentions).sum()

for username, count in mention_counts.items():
    print(f"Jumlah mention dari {username}: {count}")


In [ ]:
import pandas as pd
user_stats = {}

# Iterate over the top 5 users
for username in betweenness_top5_df['Username']:
    user_df = df[df['username'] == username]
    retweet_count = user_df['retweet_count'].sum()
    favorite_count = user_df['favorite_count'].sum()
    reply_count = user_df['reply_count'].sum()
    user_stats[username] = {
        'Retweets': retweet_count,
        'Favorites': favorite_count,
        'Replies': reply_count
    }
# Create a DataFrame from the user_stats dictionary
stats_df = pd.DataFrame.from_dict(user_stats, orient='index')

# Display the DataFrame
print(stats_df.astype(int))

## Closeness Centrality



In [ ]:
print("\nTop 5 Users by Closeness Centrality:")
print(closeness_top5_df)

In [ ]:
tweet_counts = {}
for username in closeness_top5_df['Username']:
    tweet_counts[username] = len(df[df['username'] == username])

for username, count in tweet_counts.items():
    print(f"Jumlah tweet dari {username}: {count}")

In [ ]:
# Calculate mention counts for each user in closeness_top5_df
mention_counts = {}
for username in closeness_top5_df['Username']:
    mention_counts[username] = df['mentions'].apply(lambda mentions: username in mentions).sum()

for username, count in mention_counts.items():
    print(f"Jumlah mention dari {username}: {count}")


In [ ]:
import pandas as pd
user_stats = {}

# Iterate over the top 5 users
for username in closeness_top5_df['Username']:
    user_df = df[df['username'] == username]
    retweet_count = user_df['retweet_count'].sum()
    favorite_count = user_df['favorite_count'].sum()
    reply_count = user_df['reply_count'].sum()
    user_stats[username] = {
        'Retweets': retweet_count,
        'Favorites': favorite_count,
        'Replies': reply_count
    }
# Create a DataFrame from the user_stats dictionary
stats_df = pd.DataFrame.from_dict(user_stats, orient='index')

# Display the DataFrame
print(stats_df.astype(int))

## Overall Mention Analysis

In [ ]:
# Calculate tweet count per user
tweet_counts = df['username'].value_counts()

# Display the top 5 users with the most tweets
print("Top 5 Users with the Most Tweets:")
print(tweet_counts.head(5))

In [ ]:
# Calculate mention counts
mention_counts = {}
for mentions in df['mentions']:
    for mention in mentions:
        mention_counts[mention] = mention_counts.get(mention, 0) + 1

# Sort mention counts and get top 5
sorted_mention_counts = sorted(mention_counts.items(), key=lambda x: x[1], reverse=True)
top_5_mentions = sorted_mention_counts[:5]

# Display the top 5 mentioned accounts
print("Top 5 Mentioned Accounts:")
for mention, count in top_5_mentions:
    print(f"{mention}: {count}")


In [ ]:
# Distribusi Sentimen Aktor Kunci berdasarkan Mention

def compute_sentiment_distribution_by_mention(df, actors, label_col='label (indoBERT)'):
    """
    Menghitung proporsi sentimen (positive, neutral, negative) untuk setiap aktor
    berdasarkan jumlah tweet yang menyebut (mention) aktor tersebut.
    - df         : DataFrame utama, berisi kolom 'full_text' dan kolom label_sentimen.
    - actors     : list username aktor kunci.
    - label_col  : nama kolom yang berisi label sentimen di df.
    """
    dist_list = []
    for actor in actors:
        # Filter tweet yang menyebut actor
        mention_mask = df['full_text'].astype(str).str.contains(rf'@{actor}\b', case=False, na=False)
        actor_mentions = df[mention_mask]
        total = len(actor_mentions)
        if total == 0:
            dist_list.append({
                'Username': actor,
                'Total Mentions': 0,
                'Prop_Positive': 0.0,
                'Prop_Neutral': 0.0,
                'Prop_Negative': 0.0
            })
            continue

        positives = (actor_mentions[label_col] == 'positive').sum()
        neutrals  = (actor_mentions[label_col] == 'neutral').sum()
        negatives = (actor_mentions[label_col] == 'negative').sum()

        dist_list.append({
            'Username': actor,
            'Total Mentions': total,
            'Prop_Positive': positives / total,
            'Prop_Neutral': neutrals   / total,
            'Prop_Negative': negatives / total
        })
    return pd.DataFrame(dist_list)

# Gabungkan semua aktor unik dari ketiga metrik centrality
top_degree_actors      = degree_top5_df['Username'].tolist()
top_betweenness_actors = betweenness_top5_df['Username'].tolist()
top_closeness_actors   = closeness_top5_df['Username'].tolist()
all_key_actors = list(np.unique(top_degree_actors + top_betweenness_actors + top_closeness_actors))

# Hitung distribusi sentimen berdasarkan mention untuk setiap aktor kunci
sentiment_dist_df = compute_sentiment_distribution_by_mention(
    df,
    all_key_actors,
    label_col='label (indoBERT)'
)
print("\nDistribusi Sentimen untuk Aktor Kunci (berdasarkan mention):")
print(sentiment_dist_df)

In [ ]:
# Analisis Lonjakan Sentimen akibat Retweet

def analyze_retweet_surges(df, target_actor, time_col='created_at', label_col='label (indoBERT)'):
    """
    Menampilkan lonjakan sentimen per jam dan contoh retweet pertama untuk akun kunci.
    - df            : DataFrame utama, berisi 'full_text', kolom waktu, dan kolom sentimen.
    - target_actor  : username akun kunci (misal 'nadiemmakarim').
    - time_col      : nama kolom yang memuat timestamp tweet.
    - label_col     : nama kolom sentimen di df.
    """
    # Cari tweet yang merupakan retweet ke target_actor (case-insensitive)
    pattern = rf'RT @{target_actor}\b'
    retweets = df[df['full_text'].astype(str).str.contains(pattern, case=False, na=False)].copy()

    if retweets.empty:
        print(f"\nTidak ada retweet untuk akun {target_actor}.")
        return

    # Konversi kolom waktu ke datetime jika belum
    retweets[time_col] = pd.to_datetime(retweets[time_col], errors='coerce')

    # Agregasi per jam berdasarkan label sentimen
    retweets['hour'] = retweets[time_col].dt.floor('H')
    surge_by_hour = retweets.groupby(['hour', label_col]).size().unstack(fill_value=0)

    print(f"\nLonjakan sentimen pada retweet @{target_actor} (per jam):")
    print(surge_by_hour.head(10))

    # Tampilkan contoh 5 retweet pertama
    example_retweets = retweets[['created_at', 'username', label_col, 'full_text']] \
                       .sort_values(by=time_col).head(5)
    print(f"\nContoh 5 retweet pertama ke @{target_actor}:")
    print(example_retweets.to_string(index=False))

# Panggil fungsi untuk semua aktor kunci
for actor in all_key_actors:
    analyze_retweet_surges(df, actor)